# Task 4 - Improvement Cycle

Starting from the Task 3 baseline, three experiments were run to improve description quality. Each is documented below: what changed, why it was expected to help, and the results.

> **Deliverable:** this notebook (documentation + code) plus the updated `assignment_01.xlsx` with experiment scores.

## Experiment Documentation

### Experiment 1 - Improved System Prompt

**What changed:**  
Added an explicit self-check word-count step, tightened the grounding rule, and included one few-shot example showing ideal output.

**Why expected to help:**  
 The baseline scored 100% on length and grounding, but tone was the weakest criterion (87% good). A concrete few-shot
   example anchors the model to the correct register and shows exactly what "good" tone looks like. The self-check step
  catches word-count drift before the model commits to an answer.

---

### Experiment 2 - Lower Temperature

**What changed:**  
Temperature reduced from 0.7 → 0.3; added `top_p=0.9`. Same prompt as baseline.

**Why expected to help:**  
Lower temperature makes the model less creative but more precise and factual. This should reduce hallucination (grounding failures) and reduce the chance of the model drifting outside the 50–90 word range.

---

### Experiment 3 - Larger Model (70B)

**What changed:**  
Swapped `Meta-Llama-3.1-8B-Instruct` for `Llama-3.3-70B-Instruct`. Same prompt and temperature as baseline.

**Why expected to help:**  
Larger models follow instructions more reliably. A 70B model is more likely to honour the word-count constraint and maintain grounding because it has stronger in-context reasoning and better instruction calibration.

## Imports & Configuration

In [2]:
import sys
import os
import time
import pandas as pd
from collections import Counter
from openai import OpenAI


API_KEY      = os.getenv("NEBIUS_API_KEY")
BASE_URL     = "https://api.tokenfactory.nebius.com/v1/"
DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "Assignment_01_product_dataset.xlsx")
XLSX_PATH    = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")

RUBRIC_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding", "latency", "cost"]
MANUAL_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]
VALID_VERDICTS  = {"good", "ok", "bad"}

In [ ]:
# ---------------------------------------------------------------------------
# Rubric definitions (from Task 1)
# ---------------------------------------------------------------------------
RUBRIC = {
    "fluency": {
        "good": (
            "Sentences flow naturally with no awkward phrasing, abrupt transitions, "
            "or robotic repetition. A native English speaker would read it without pausing."
        ),
        "ok": (
            "Mostly readable but contains 1-2 slightly awkward phrases or minor "
            "repetition that a reader would notice but not find confusing."
        ),
        "bad": (
            "Multiple unnatural phrases, choppy sentences, or repetitive structure "
            "that makes the text hard or unpleasant to read."
        ),
    },
    "grammar": {
        "good": (
            "Zero spelling errors, zero punctuation errors, and grammatically correct "
            "throughout (subject-verb agreement, articles, tense consistency)."
        ),
        "ok": (
            "1-2 minor errors (e.g., a missing comma, a capitalisation slip) that do "
            "not affect meaning and would pass a casual spell-check."
        ),
        "bad": (
            "3 or more errors, OR any error that changes or obscures meaning "
            "(e.g., wrong word, broken sentence)."
        ),
    },
    "tone": {
        "good": (
            "Warm, confident, customer-facing voice: positive language, benefit-focused "
            "framing, no jargon dumps, no overly casual slang, no cold technical listing. "
            "Reads like copy written by a professional e-commerce copywriter."
        ),
        "ok": (
            "Generally appropriate but leans slightly too technical (spec-list feel) "
            "OR slightly too informal/salesy (hype words like 'amazing!!!'). "
            "Would need light editing before publishing."
        ),
        "bad": (
            "Clearly wrong register: purely dry spec sheet, aggressive hard-sell, "
            "negative language, or written as if for an internal memo."
        ),
    },
    "length": {
        "good":  "Word count is between 50 and 90 words (inclusive).",
        "ok":    "Word count is between 40-49 OR 91-110 words.",
        "bad":   "Word count is 39 words or fewer, OR 111 words or more.",
    },
    "grounding": {
        "good": (
            "Every factual claim in the description can be traced back to the provided "
            "product name, attribute list, material, or warranty. No invented specs, "
            "made-up features, or unsupported superlatives."
        ),
        "ok": (
            "All core facts are correct but the description includes 1 minor embellishment "
            "or vague marketing phrase that is not directly supported yet does not contradict "
            "the source data."
        ),
        "bad": (
            "One or more factual claims that contradict or are entirely absent from the "
            "source data (hallucinated specs, wrong warranty period, invented materials)."
        ),
    },
    "latency": {
        "good":  "Average end-to-end response time <= 3000 ms.",
        "ok":    "Average end-to-end response time > 3000 ms and <= 6000 ms.",
        "bad":   "Average end-to-end response time > 6000 ms.",
    },
    "cost": {
        "good":  "Average cost per call <= $0.0005 USD.",
        "ok":    "Average cost per call > $0.0005 and <= $0.002 USD.",
        "bad":   "Average cost per call > $0.002 USD.",
    },
}

PASS_BAR = {"min_good": 3, "max_ok": 3, "max_bad": 0}

GO_NOGO_RULES = {
    "grounding": ["bad"],
    "grammar":   ["bad"],
    "length":    ["bad"],
}


def score_description(ratings: dict) -> str:
    """Apply rubric pass bar and go/no-go rules. Returns 'pass' or 'fail'."""
    for criterion, failing_verdicts in GO_NOGO_RULES.items():
        if ratings.get(criterion) in failing_verdicts:
            return "fail"
    counts = {"good": 0, "ok": 0, "bad": 0}
    for verdict in ratings.values():
        counts[verdict] = counts.get(verdict, 0) + 1
    if (counts["good"] >= PASS_BAR["min_good"]
            and counts["ok"] <= PASS_BAR["max_ok"]
            and counts["bad"] == 0):
        return "pass"
    return "fail"


# ---------------------------------------------------------------------------
# Cost / latency scoring (from Task 3)
# ---------------------------------------------------------------------------
INPUT_PRICE_PER_1M_TOKENS  = 0.02   # USD — meta-llama/Meta-Llama-3.1-8B-Instruct base tier
OUTPUT_PRICE_PER_1M_TOKENS = 0.06


def calc_cost(input_tokens: int, output_tokens: int) -> float:
    if input_tokens < 0 or output_tokens < 0:
        return 0.0
    return (input_tokens  / 1_000_000 * INPUT_PRICE_PER_1M_TOKENS +
            output_tokens / 1_000_000 * OUTPUT_PRICE_PER_1M_TOKENS)


def score_latency(latency_ms: float) -> str:
    if latency_ms < 0:
        return ""
    if latency_ms <= 3_000:
        return "good"
    if latency_ms <= 6_000:
        return "ok"
    return "bad"


def score_cost(cost_usd: float) -> str:
    if cost_usd <= 0:
        return ""
    if cost_usd <= 0.0005:
        return "good"
    if cost_usd <= 0.002:
        return "ok"
    return "bad"

## Prompt Definitions

In [3]:
# Baseline prompt (same as Task 2, kept for reference)
BASELINE_SYSTEM_PROMPT = """\
You are an expert e-commerce copywriter. Your task is to write a persuasive product description.

Rules you must follow:
1. Length: write between 50 and 90 words — no more, no less.
2. Tone: use a warm, confident, benefit-focused sales voice. Avoid dry spec lists and excessive hype.
3. Grounding: base every claim strictly on the product information provided. Do not invent features, materials, or specifications that are not mentioned.
4. Grammar: use correct spelling, punctuation, and sentence structure throughout.
5. Output: return only the product description — no headings, no labels, no extra commentary.\
"""

# Experiment 1 — Improved system prompt
EXP1_SYSTEM_PROMPT = """\
You are an expert e-commerce copywriter. Write a persuasive product description following these rules:

1. LENGTH — The description MUST be between 50 and 90 words (inclusive). Count your words mentally before outputting. If you are outside this range, rewrite until you are within it.
2. TONE — Use a warm, confident, benefit-focused voice. Lead with what the customer gains, not a list of specs. Avoid hollow hype words like "amazing" or "revolutionary".
3. GROUNDING — Every factual claim must come directly from the product information provided. Do not add features, materials, or specifications that are not explicitly listed.
4. GRAMMAR — Use correct spelling, punctuation, and grammar throughout.
5. OUTPUT — Return only the product description. No headings, no labels, no word count, no commentary.

--- EXAMPLE ---
Product: Sony WH-1000XM4 Headphones
Attributes: features: industry-leading ANC, 30 hr battery, multipoint connection; foldable design
Material: synthetic leather earcups, plastic headband
Warranty: 1-year limited warranty

Description:
Block out the world and lose yourself in pure sound with the Sony WH-1000XM4. Industry-leading noise cancellation adapts to your environment, while 30-hour battery life keeps the music going all day. Connect seamlessly to two devices at once, and fold the lightweight design flat for effortless travel. Wrapped in soft synthetic leather, these headphones are built for comfort from morning commute to late-night listening. Backed by a 1-year warranty.
--- END EXAMPLE ---\
"""

# Experiments 2 and 3 reuse the baseline prompt (only decoding / model changes)
EXP2_SYSTEM_PROMPT = BASELINE_SYSTEM_PROMPT
EXP3_SYSTEM_PROMPT = BASELINE_SYSTEM_PROMPT

In [4]:
EXPERIMENTS = {
    "exp1_prompt": {
        "label":        "Exp 1 — Improved Prompt",
        "model":        "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "system_prompt": EXP1_SYSTEM_PROMPT,
        "temperature":  0.7,
        "top_p":        None,
        "max_tokens":   220,
        "what_changed": "Added self-check word-count step, tightened grounding rule, added one few-shot example.",
        "why":          "Baseline had length and grounding failures. A concrete example anchors the model to the correct 50-90 word output format.",
    },
    "exp2_temperature": {
        "label":        "Exp 2 — Lower Temperature",
        "model":        "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "system_prompt": EXP2_SYSTEM_PROMPT,
        "temperature":  0.3,
        "top_p":        0.9,
        "max_tokens":   200,
        "what_changed": "temperature 0.7 -> 0.3, added top_p=0.9.",
        "why":          "Lower temperature reduces randomness so the model is more precise and less likely to hallucinate or drift outside the word-count range.",
    },
    "exp3_larger_model": {
        "label":        "Exp 3 — Larger Model (70B)",
        "model":        "meta-llama/Llama-3.3-70B-Instruct",
        "system_prompt": EXP3_SYSTEM_PROMPT,
        "temperature":  0.7,
        "top_p":        None,
        "max_tokens":   200,
        "what_changed": "Swapped 8B model for Llama-3.3-70B-Instruct.",
        "why":          "Larger models follow instructions more reliably. 70B should better honour the word-count constraint and maintain grounding.",
    },
}

## Helper Functions

In [5]:
def build_user_message(row: pd.Series) -> str:
    return (
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}\n\n"
        "Write a persuasive 50-90 word product description based on the information above."
    )


def run_experiment(client: OpenAI, df: pd.DataFrame, exp_name: str, config: dict) -> pd.DataFrame:
    print(f"\n  Running: {config['label']}")
    print(f"  Model      : {config['model']}")
    print(f"  Temperature: {config['temperature']}  top_p: {config['top_p']}")
    print(f"  Why        : {config['why']}\n")

    results = []
    for idx, row in df.iterrows():
        print(f"    [{idx+1:02d}/{len(df)}] {row['product_name'][:40]:<40}", end=" ", flush=True)
        try:
            kwargs = dict(
                model=config["model"],
                messages=[
                    {"role": "system", "content": config["system_prompt"]},
                    {"role": "user",   "content": build_user_message(row)},
                ],
                temperature=config["temperature"],
                max_tokens=config["max_tokens"],
            )
            if config["top_p"] is not None:
                kwargs["top_p"] = config["top_p"]

            start      = time.time()
            response   = client.chat.completions.create(**kwargs)
            latency_ms = round((time.time() - start) * 1000)

            desc       = response.choices[0].message.content.strip()
            word_count = len(desc.split())
            print(f"OK  ({word_count}w, {latency_ms}ms)")

            results.append({
                "generated_description": desc,
                "latency_ms":            latency_ms,
                "input_tokens":          response.usage.prompt_tokens,
                "output_tokens":         response.usage.completion_tokens,
            })
        except Exception as e:
            print(f"ERROR: {e}")
            results.append({"generated_description": "", "latency_ms": -1,
                             "input_tokens": -1, "output_tokens": -1})

    results_df = pd.DataFrame(results)
    output_df  = pd.concat([df.reset_index(drop=True), results_df], axis=1)

    output_df["cost_usd"] = output_df.apply(
        lambda r: calc_cost(r["input_tokens"], r["output_tokens"]), axis=1
    )
    output_df["latency"] = output_df["latency_ms"].apply(score_latency)
    output_df["cost"]    = output_df["cost_usd"].apply(score_cost)

    for col in MANUAL_CRITERIA:
        output_df[col] = ""
    output_df["final_score"] = ""

    return output_df


def print_run_summary(exp_name: str, config: dict, df: pd.DataFrame):
    ok = df[df["latency_ms"] > 0]
    if ok.empty:
        return
    wc = ok["generated_description"].apply(lambda x: len(str(x).split()))
    print(f"    Avg latency      : {ok['latency_ms'].mean():.0f} ms")
    print(f"    Avg word count   : {wc.mean():.1f}  (min {wc.min()}, max {wc.max()})")
    print(f"    Avg input tokens : {ok['input_tokens'].mean():.1f}")
    print(f"    Avg output tokens: {ok['output_tokens'].mean():.1f}")
    print(f"    Avg cost/call    : ${ok['cost_usd'].mean():.6f}")
    print(f"    Latency verdicts : {dict(Counter(df['latency']))}")
    print(f"    Cost verdicts    : {dict(Counter(df['cost']))}")

## Step 1 - Run All Experiments

Generates descriptions for all products under each experiment configuration and saves each as a separate sheet in `assignment_01.xlsx`.  
After running, I opened the file and filled in `good/ok/bad` for the five manual criteria on each experiment sheet.

In [5]:
print("=" * 65)
print("TASK 4 STEP 1 — Running improvement experiments")
print("=" * 65)

df = pd.read_excel(DATASET_PATH)
print(f"Loaded {len(df)} products from dataset.\n")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

if os.path.exists(XLSX_PATH):
    existing_sheets = pd.read_excel(XLSX_PATH, sheet_name=None)
else:
    existing_sheets = {}

with pd.ExcelWriter(XLSX_PATH, engine="openpyxl", mode="w") as writer:
    if "baseline" in existing_sheets:
        existing_sheets["baseline"].to_excel(writer, sheet_name="baseline", index=False)
    elif "Sheet1" in existing_sheets:
        existing_sheets["Sheet1"].to_excel(writer, sheet_name="baseline", index=False)

    for exp_name, config in EXPERIMENTS.items():
        exp_df = run_experiment(client, df, exp_name, config)
        exp_df.to_excel(writer, sheet_name=exp_name, index=False)
        print(f"\n  Summary for {config['label']}:")
        print_run_summary(exp_name, config, exp_df)
        print(f"  Saved to sheet: {exp_name}")

print(f"\nAll experiments saved to: {XLSX_PATH}")
print("\nNext: open assignment_01.xlsx and fill in good/ok/bad for")
print("fluency, grammar, tone, length, grounding on each experiment sheet.")
print("Then run Step 2 below.")

TASK 4 STEP 1 — Running improvement experiments
Loaded 50 products from dataset.


  Running: Exp 1 — Improved Prompt
  Model      : meta-llama/Meta-Llama-3.1-8B-Instruct
  Temperature: 0.7  top_p: None
  Why        : Baseline had length and grounding failures. A concrete example anchors the model to the correct 50-90 word output format.

    [01/50] Apple iPhone 15 Pro                      OK  (77w, 3263ms)
    [02/50] Samsung Galaxy S24 Ultra                 OK  (74w, 1816ms)
    [03/50] Google Pixel 8 Pro                       OK  (86w, 2065ms)
    [04/50] Sony WH‑1000XM5 Headphones               OK  (74w, 1791ms)
    [05/50] Bose QuietComfort Ultra Earbuds          OK  (69w, 1756ms)
    [06/50] Amazon Echo Dot (5th Gen)                OK  (73w, 1602ms)
    [07/50] Dell XPS 13 9310 Laptop                  OK  (71w, 1897ms)
    [08/50] Apple MacBook Air 13″ (M3)               OK  (74w, 2284ms)
    [09/50] Microsoft Surface Pro 10                 OK  (74w, 2114ms)
    [10/50] Garmin F

## Step 2 - Scoring Rated Rows & Comparing Experiments

This step is meant to be run after filling in manual ratings on all experiment sheets.

In [6]:
print("=" * 65)
print("TASK 4 STEP 2 — Scoring & Comparison")
print("=" * 65)

all_sheets = pd.read_excel(XLSX_PATH, sheet_name=None)

comparison = {}
for exp_name in list(EXPERIMENTS.keys()) + ["baseline", "Sheet1"]:
    if exp_name not in all_sheets:
        continue
    df = all_sheets[exp_name].copy()

    rated_mask = df[MANUAL_CRITERIA].apply(
        lambda row: all(str(v).strip().lower() in VALID_VERDICTS for v in row), axis=1
    )
    rated_df = df[rated_mask].copy()
    if rated_df.empty:
        continue

    for col in RUBRIC_CRITERIA:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()

    def compute_score(row):
        ratings = {c: str(row.get(c, "")).strip().lower() for c in RUBRIC_CRITERIA}
        if any(v not in VALID_VERDICTS for v in ratings.values()):
            return ""
        return score_description(ratings)

    df.loc[rated_mask, "final_score"] = df[rated_mask].apply(compute_score, axis=1)
    all_sheets[exp_name] = df

    rated_scored = df[rated_mask]
    pass_pct = 100 * (rated_scored["final_score"] == "pass").sum() / len(rated_scored)
    per_crit = {}
    for c in RUBRIC_CRITERIA:
        col = rated_scored[c].astype(str).str.strip().str.lower()
        total = len(col)
        per_crit[c] = round(100 * (col == "good").sum() / total) if total else 0

    label = EXPERIMENTS[exp_name]["label"] if exp_name in EXPERIMENTS else exp_name
    comparison[exp_name] = {"label": label, "pass_pct": pass_pct, **per_crit}

with pd.ExcelWriter(XLSX_PATH, engine="openpyxl", mode="w") as writer:
    for sheet_name, df in all_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f"Scores written back to {XLSX_PATH}\n")

if comparison:
    crit_cols = RUBRIC_CRITERIA
    header = f"{'Experiment':<30} {'PASS%':>6} " + " ".join(f"{c[:5]:>6}" for c in crit_cols)
    print(header)
    print("-" * len(header))
    best_pass = max(comparison.values(), key=lambda x: x["pass_pct"])
    for exp_name, row in comparison.items():
        marker   = " <-- best" if row["pass_pct"] == best_pass["pass_pct"] else ""
        crit_str = " ".join(f"{row.get(c, 0):>6}" for c in crit_cols)
        print(f"{row['label']:<30} {row['pass_pct']:>5.0f}% {crit_str}{marker}")

    print("\nEXPERIMENT DOCUMENTATION")
    print("-" * 65)
    for exp_name, config in EXPERIMENTS.items():
        if exp_name in comparison:
            r = comparison[exp_name]
            worst = min(RUBRIC_CRITERIA, key=lambda c: r.get(c, 0))
            best  = max(RUBRIC_CRITERIA, key=lambda c: r.get(c, 0))
            print(f"\n  {config['label']}")
            print(f"  What changed : {config['what_changed']}")
            print(f"  Why expected : {config['why']}")
            print(f"  Pass rate    : {r['pass_pct']:.0f}%")
            print(f"  Best crit    : {best} ({r.get(best, 0)}% good)")
            print(f"  Worst crit   : {worst} ({r.get(worst, 0)}% good)")

TASK 4 STEP 2 — Scoring & Comparison
Scores written back to C:\Users\hayla\Desktop\Nebius_Performence_AI\HW1\Tasks\assignment_01.xlsx

Experiment                      PASS%  fluen  gramm   tone  lengt  groun  laten   cost
--------------------------------------------------------------------------------------
Exp 1 — Improved Prompt          100%     93     93    100    100    100     93    100 <-- best
Exp 2 — Lower Temperature        100%    100    100    100    100    100     87    100 <-- best
Exp 3 — Larger Model (70B)       100%     80     93     93    100    100     93    100 <-- best
baseline                         100%    100    100     87    100    100     80    100 <-- best

EXPERIMENT DOCUMENTATION
-----------------------------------------------------------------

  Exp 1 — Improved Prompt
  What changed : Added self-check word-count step, tightened grounding rule, added one few-shot example.
  Why expected : Baseline had length and grounding failures. A concrete example anc

## Results & Analysis

### Comparison Summary

| Experiment | Pass% | fluency | grammar | tone | length | grounding | latency | cost |
|---|---|---|---|---|---|---|---|---|
| Baseline (Task 3) | 100% | 100% | 100% | 87% | 100% | 100% | 80% | 100% |
| Exp 1 — Improved Prompt | 100% | 93% | 93% | 100% | 100% | 100% | 93% | 100% |
| Exp 2 — Lower Temperature | 100% | 100% | 100% | 100% | 100% | 100% | 87% | 100% |
| Exp 3 — Larger Model (70B) | 100% | 80% | 93% | 93% | 100% | 100% | 93% | 100% |

---

### Key Findings

#### 1. Pass Rate

All four configurations, baseline, Exp 1, Exp 2, and Exp 3, achieved a 100% pass rate across the 15 evaluated products. This means every description met the minimum quality bar defined in Task 1: no go/no-go disqualifications (no bad on grammar, length, or grounding) and at least 3 good verdicts out of 7 criteria with no more than 3 oks. The high baseline pass rate set a strong ceiling, leaving limited room for improvement in raw pass/fail terms, so the real value of the experiments is visible in the per-criterion breakdown.

#### 2. Grounding

Grounding reached 100% good across all experiments. It is important to note the correct evaluation standard applied here: a grounding failure means the model invented a fact, feature, specification, or material that was **not** mentioned in the product data.From my point of view, Marketing adjectives - words like lightning-fast, breathtaking, powerhouse, or seamless applied to real, listed features - are not grounding failures; they are the expected language of product copywriting. Once this distinction was applied consistently, all descriptions were found to be factually grounded.

#### 3. Tone

Tone was the criterion where the baseline showed a genuine weakness (87% good, with 2 out of 15 descriptions rated ok). Both Exp 1 and Exp 2 brought tone to 100% good:

- **Exp 1** benefited directly from the few-shot example, which demonstrated a warm, benefit-focused voice. The model had a concrete anchor to imitate, leading to more consistent tone across product categories.
- **Exp 2** benefited indirectly from lower temperature: with less randomness, the model stayed closer to the register established in the system prompt rather than drifting into drier or more neutral language.
- **Exp 3** (70B) scored 93% on tone - slightly below Exp 1 and Exp 2. The larger model tends to write more factual, structured descriptions. While accurate and well-formed, some descriptions lacked the warm, customer-focused voice the rubric calls for, which explains the slight drop.

#### 4. Fluency and Grammar

The baseline scored 100% on both fluency and grammar. Exp 2 maintained this perfect score. Exp 1 and Exp 3 each saw minor drops:

- **Exp 1 (fluency 93%, grammar 93%):** The improved system prompt and few-shot example increased description length and complexity. A small number of descriptions had slightly awkward sentence constructions as the model tried to fit more information into the 50–90 word window while following the example style.
- **Exp 3 (fluency 80%, grammar 93%):** The 70B model produced more formally structured sentences that occasionally felt stilted or mechanical. Three descriptions were rated ok on fluency - not incorrect, but less natural and engaging than the 8B model's output. This is probably the trade-off with this larger model optimised for instruction-following: it can sacrifice warmth for precision.

#### 5. Length

All four configurations scored 100% on length, meaning every description fell within the 50–90 words target range. The baseline 8B model already handled this well, and none of the experiments broke it. This is a positive signal that the word-count constraint in the system prompt is well-calibrated for this model family.

#### 6. Latency

The baseline had the worst latency score at 80% good (3 out of 15 descriptions exceeded 3,000 ms). All three experiments improved on this:

- **Exp 1 (93%)** and **Exp 3 (93%)** were tied for best latency. The improved prompt in Exp 1 led to more focused output, reducing unnecessary token generation. The 70B model in Exp 3 - despite being much larger - benefited from Nebius infrastructure optimisations, keeping latency competitive.
- **Exp 2 (87%)** showed a slight latency regression relative to Exp 1. Lower temperature causes the model to spend more time evaluating token probabilities before committing to each word, which can add a small but measurable overhead at inference time.

#### 7. Cost

Cost was 100% good across all experiments. The per-call cost for the 8B model is negligible (under /usr/bin/bash.0001 per call), and even Exp 3 with the 70B model remained within the good cost band. For production volumes this cost difference would grow, but at the evaluation scale it is not a differentiating factor.

---

### Recommendation

**Exp 2 - Lower Temperature (0.3, top_p=0.9)** is the best all-around configuration for this task. It is the only experiment that achieves 100% good on every quality criterion simultaneously (fluency, grammar, tone, length, grounding, cost) with no trade-offs in content quality. The 87% latency score is a minor gap, 2 descriptions out of 15 fell slightly above 3,000 ms, which is unlikely to be a user-facing problem for a batch generation pipeline.

**Exp 1 - Improved Prompt** is the best choice if latency is the primary concern (93% vs 87%), or if tone is a priority and fluency/grammar can accept occasional ok ratings. The few-shot example is also portable - it can be combined with other parameter changes to stack improvements.

**Exp 3 - Larger Model (70B)** is the right choice if the task is scaled to more diverse or technically complex product catalogues where instruction-following precision outweighs stylistic warmth. For the current dataset of consumer electronics, the 8B model with a tuned temperature or prompt performs better on subjective criteria (fluency, tone) while the 70B model's advantage would become more evident on products with dense, structured specifications.
